In [2]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]

In [3]:
print("--- Question 1 ---")
missing = df.isnull().sum()
print("Missing values per column:\n", missing)

--- Question 1 ---
Missing values per column:
 engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [4]:
print("\n--- Question 2 ---")
horsepower_median = df['horsepower'].median()
print(f"Median of horsepower: {horsepower_median}")


--- Question 2 ---
Median of horsepower: 254.0


In [5]:
def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy().reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy().reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].copy().reset_index(drop=True)

    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values

    del df_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']

    return df_train, df_val, df_test, y_train, y_val, y_test

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

def prepare_X(df, fill_value):
    df_num = df.fillna(fill_value)
    return df_num.values


In [6]:
print("\n--- Question 3 ---")
df_train, df_val, df_test, y_train, y_val, y_test = split_data(df, seed=42)

# الخيار الأول: التعبئة بـ 0
X_train_0 = prepare_X(df_train, 0)
w0, w = train_linear_regression(X_train_0, y_train)
X_val_0 = prepare_X(df_val, 0)
y_pred_0 = w0 + X_val_0.dot(w)
rmse_0 = round(rmse(y_val, y_pred_0), 3)

# الخيار الثاني: التعبئة بالمتوسط الحسابي (تأكد من حسابه من بيانات التدريب فقط!)
mean_val = df_train['horsepower'].mean()
X_train_mean = prepare_X(df_train, mean_val)
w0, w = train_linear_regression(X_train_mean, y_train)
X_val_mean = prepare_X(df_val, mean_val)
y_pred_mean = w0 + X_val_mean.dot(w)
rmse_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"RMSE (Fill with 0): {rmse_0}")
print(f"RMSE (Fill with mean): {rmse_mean}")



--- Question 3 ---
RMSE (Fill with 0): 2.205
RMSE (Fill with mean): 2.202


In [7]:
print("\n--- Question 4 ---")
r_list = [0, 0.01, 0.1, 1, 5, 10, 100]
X_train_0 = prepare_X(df_train, 0)
X_val_0 = prepare_X(df_val, 0)

for r in r_list:
    w0, w = train_linear_regression_reg(X_train_0, y_train, r=r)
    y_pred = w0 + X_val_0.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:<5} -> RMSE = {score}")


--- Question 4 ---
r = 0     -> RMSE = 2.2053
r = 0.01  -> RMSE = 2.2058
r = 0.1   -> RMSE = 2.2241
r = 1     -> RMSE = 2.3492
r = 5     -> RMSE = 2.4094
r = 10    -> RMSE = 2.4195
r = 100   -> RMSE = 2.4292


In [8]:
print("\n--- Question 5 ---")
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
scores = []

for s in seeds:
    dt, dv, dtest, yt, yv, ytest = split_data(df, seed=s)
    Xt = prepare_X(dt, 0)
    Xv = prepare_X(dv, 0)
    w0, w = train_linear_regression(Xt, yt)
    ypred = w0 + Xv.dot(w)
    scores.append(rmse(yv, ypred))

std_val = round(np.std(scores), 3)
print(f"Standard Deviation across seeds: {std_val}")


--- Question 5 ---
Standard Deviation across seeds: 0.029


In [9]:
print("\n--- Question 6 ---")
dt, dv, dtest, yt, yv, ytest = split_data(df, seed=9)

df_full_train = pd.concat([dt, dv]).reset_index(drop=True)
y_full_train = np.concatenate([yt, yv])

X_full = prepare_X(df_full_train, 0)
X_test = prepare_X(dtest, 0)

w0, w = train_linear_regression_reg(X_full, y_full_train, r=0.001)
y_pred = w0 + X_test.dot(w)
test_rmse = round(rmse(ytest, y_pred), 3)
print(f"Final Test RMSE (seed=9, r=0.001): {test_rmse}")


--- Question 6 ---
Final Test RMSE (seed=9, r=0.001): 2.236
